In [15]:
data_dir = "/kaggle/input/datasets/namunaregmi/assessment-vegetables/Vegetable Images"

train_dir = data_dir + "/train"
val_dir = data_dir + "/validation"
test_dir = data_dir + "/test"

In [16]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

from sklearn.metrics import classification_report

**Data Generators with Augmentation**

In [17]:
img_size = (224, 224)
batch_size = 32

# Augmentation ONLY for training
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_data.num_classes
print("Classes:", num_classes)

Found 15000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.
Classes: 15


**Baseline Model**

Model Architecture

In [18]:
model_baseline = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),

    layers.Dense(num_classes, activation='softmax')
])

model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_baseline.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,179,791 (42.65 MB)

 Trainable params: 11,179,791 (42.65 MB)

 Non-trainable params: 0 (0.00 B)

Train Baseline

In [19]:
start = time.time()

history_baseline = model_baseline.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

baseline_time = time.time() - start
print("Baseline Training Time:", baseline_time)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 367ms/step - accuracy: 0.3444 - loss: 1.9471 - val_accuracy: 0.7767 - val_loss: 0.7313
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 367ms/step - accuracy: 0.7505 - loss: 0.7694 - val_accuracy: 0.8250 - val_loss: 0.5809
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 367ms/step - accuracy: 0.8551 - loss: 0.4655 - val_accuracy: 0.8847 - val_loss: 0.3981
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 174s 372ms/step - accuracy: 0.9054 - loss: 0.2986 - val_accuracy: 0.8653 - val_loss: 0.4586
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 375ms/step - accuracy: 0.9218 - loss: 0.2324 - val_accuracy: 0.9373 - val_loss: 0.2207
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 177s 377ms/step - accuracy: 0.9440 - loss: 0.1778 - val_accuracy: 0.9537 - val_loss: 0.1541
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 366ms/step - accuracy: 0.9480 - loss: 0.1562 - val_accuracy: 0.9307 - val_loss: 0.2589
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 173s 368ms/step - accuracy: 0.9567 -

Plot Loss and Accuracy

In [21]:
loss, acc = model_baseline.evaluate(test_data)
print("Baseline Accuracy:", acc)

y_pred = np.argmax(model_baseline.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.9889 - loss: 0.0436
Baseline Accuracy: 0.9826666712760925
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       200
           1       0.99      0.99      0.99       200
           2       0.99      0.99      0.99       200
           3       0.99      0.99      0.99       200
           4       0.94      0.99      0.96       200
           5       0.98      0.95      0.97       200
           6       0.99      1.00      0.99       200
           7       0.99      0.99      0.99       200
           8       0.94      0.98      0.96       200
           9       0.99      0.99      0.99       200
          10       0.99      0.97      0.98       200
          11       1.00      0.99      0.99       200
          12       0.99      0.94      0.96       200
          13       1.00      0.99      0.99       200
          14       0.97      0.97    

Evaluation

In [22]:
baseline_loss, baseline_acc = model_baseline.evaluate(test_data)
print("Baseline Accuracy:", baseline_acc)

y_pred = np.argmax(model_baseline.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.9889 - loss: 0.0436
Baseline Accuracy: 0.9826666712760925
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       200
           1       0.99      0.99      0.99       200
           2       0.99      0.99      0.99       200
           3       0.99      0.99      0.99       200
           4       0.94      0.99      0.96       200
           5       0.98      0.95      0.97       200
           6       0.99      1.00      0.99       200
           7       0.99      0.99      0.99       200
           8       0.94      0.98      0.96       200
           9       0.99      0.99      0.99       200
          10       0.99      0.97      0.98       200
          11       1.00      0.99      0.99       200
          12       0.99      0.94      0.96       200
          13       1.00      0.99      0.99       200
          14       0.97      0.97    

**Deeper Model**

Architecture with Regularization

In [23]:
model_deep = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation='softmax')
])

model_deep.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 15)             │         1,935 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,862,607 (37.62 MB)

 Trainable params: 9,861,647 (37.62 MB)

 Non-trainable params: 960 (3.75 KB)

Training

In [24]:
start = time.time()

history_deep = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

deep_time = time.time() - start
print("Deep Model Training Time:", deep_time)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 214s 439ms/step - accuracy: 0.1634 - loss: 5.3976 - val_accuracy: 0.2753 - val_loss: 2.3178
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 199s 424ms/step - accuracy: 0.2228 - loss: 2.4446 - val_accuracy: 0.2877 - val_loss: 2.2203
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 196s 417ms/step - accuracy: 0.2845 - loss: 2.1987 - val_accuracy: 0.4283 - val_loss: 1.7743
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 414ms/step - accuracy: 0.3387 - loss: 2.0100 - val_accuracy: 0.5240 - val_loss: 1.4954
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 193s 411ms/step - accuracy: 0.4065 - loss: 1.7779 - val_accuracy: 0.4393 - val_loss: 1.6732
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 191s 407ms/step - accuracy: 0.4300 - loss: 1.7051 - val_accuracy: 0.4883 - val_loss: 1.8828
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 193s 412ms/step - accuracy: 0.4665 - loss: 1.5744 - val_accuracy: 0.3427 - val_loss: 2.6256
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 201s 428ms/step - accuracy: 0.5120 -

Evaluation

In [25]:
deep_loss, deep_acc = model_deep.evaluate(test_data)
print("Deep Model Accuracy:", deep_acc)

y_pred = np.argmax(model_deep.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.8083 - loss: 0.7049
Deep Model Accuracy: 0.7806666493415833
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step
              precision    recall  f1-score   support

           0       0.96      0.77      0.85       200
           1       0.99      0.90      0.94       200
           2       0.45      0.69      0.55       200
           3       0.49      0.87      0.63       200
           4       0.75      0.96      0.84       200
           5       0.60      0.82      0.69       200
           6       0.92      0.98      0.95       200
           7       0.97      0.92      0.94       200
           8       0.97      0.45      0.61       200
           9       0.85      0.91      0.87       200
          10       0.52      0.06      0.10       200
          11       0.92      0.94      0.93       200
          12       0.92      0.92      0.92       200
          13       0.89      0.85      0.87       200
          14       0.95      0.67

Comparison

In [26]:
print("Baseline Accuracy:", baseline_acc)
print("Deep Accuracy:", deep_acc)

print("Baseline Time:", baseline_time)
print("Deep Time:", deep_time)

Baseline Accuracy: 0.9826666712760925
Deep Accuracy: 0.7806666493415833
Baseline Time: 2671.8814692497253
Deep Time: 2914.298819065094


Optimizer Analysis (SGD vs Adam)

SGD

In [27]:
model_deep.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_sgd = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 208s 433ms/step - accuracy: 0.7659 - loss: 0.6907 - val_accuracy: 0.6733 - val_loss: 1.6181
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 397ms/step - accuracy: 0.7786 - loss: 0.6254 - val_accuracy: 0.8657 - val_loss: 0.5051
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 184s 392ms/step - accuracy: 0.7881 - loss: 0.6150 - val_accuracy: 0.8673 - val_loss: 0.4108
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.7899 - loss: 0.5958 - val_accuracy: 0.8973 - val_loss: 0.3047
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 197s 420ms/step - accuracy: 0.7979 - loss: 0.5658 - val_accuracy: 0.8977 - val_loss: 0.3524
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.7995 - loss: 0.5610 - val_accuracy: 0.8970 - val_loss: 0.3097
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.7971 - loss: 0.5552 - val_accuracy: 0.5923 - val_loss: 2.8274
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.8066 -

Adam

In [28]:
model_deep.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_adam = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 209s 430ms/step - accuracy: 0.7936 - loss: 0.6120 - val_accuracy: 0.8160 - val_loss: 0.8231
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.8042 - loss: 0.5965 - val_accuracy: 0.7790 - val_loss: 0.8958
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 190s 405ms/step - accuracy: 0.8087 - loss: 0.5935 - val_accuracy: 0.8393 - val_loss: 0.5568
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 189s 402ms/step - accuracy: 0.8264 - loss: 0.4961 - val_accuracy: 0.8207 - val_loss: 0.5829
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.8306 - loss: 0.5088 - val_accuracy: 0.9043 - val_loss: 0.2977
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.8372 - loss: 0.4675 - val_accuracy: 0.8230 - val_loss: 0.4764
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 410ms/step - accuracy: 0.8448 - loss: 0.4478 - val_accuracy: 0.9090 - val_loss: 0.2807
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.8528 -

**Transfer Learning**

Load Model

In [29]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

Preprocessing Change

In [32]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

Load Pretrained Model

In [33]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Freeze Base Model (Feature Extraction)

In [35]:
for layer in base_model.layers:
    layer.trainable = False

Add Custom Classification Layers

In [38]:
from tensorflow.keras import layers, Model

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(num_classes, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

Compile Model

In [39]:
model_mobilenet.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_4[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,423,887 (9.25 MB)

 Trainable params: 165,903 (648.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Train (Feature Extraction Phase)

In [40]:
start = time.time()

history_mn = model_mobilenet.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

mn_time = time.time() - start
print("MobileNetV2 Training Time:", mn_time)

Epoch 1/10


2026-05-06 11:24:18.164898: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:24:18.303302: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


426/469 ━━━━━━━━━━━━━━━━━━━━ 17s 407ms/step - accuracy: 0.8097 - loss: 0.6466

2026-05-06 11:27:23.658174: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:27:23.801398: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:27:23.938344: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


469/469 ━━━━━━━━━━━━━━━━━━━━ 238s 469ms/step - accuracy: 0.8201 - loss: 0.6117 - val_accuracy: 0.9927 - val_loss: 0.0246
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.9771 - loss: 0.0776 - val_accuracy: 0.9957 - val_loss: 0.0169
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 185s 395ms/step - accuracy: 0.9844 - loss: 0.0515 - val_accuracy: 0.9967 - val_loss: 0.0106
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 396ms/step - accuracy: 0.9877 - loss: 0.0408 - val_accuracy: 0.9983 - val_loss: 0.0061
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 187s 399ms/step - accuracy: 0.9888 - loss: 0.0345 - val_accuracy: 0.9987 - val_loss: 0.0057
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 396ms/step - accuracy: 0.9895 - loss: 0.0349 - val_accuracy: 0.9977 - val_loss: 0.0095
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.9916 - loss: 0.0256 - val_accuracy: 0.9990 - val_loss: 0.0064
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 191s 408ms/step - accuracy: 0.9924 - loss: 0.02

**Fine-Tuning**

In [41]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

**Recompile with LOW LR**

In [42]:
from tensorflow.keras.optimizers import Adam

model_mobilenet.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Fine-Tune Training

In [43]:
history_ft = model_mobilenet.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 235s 470ms/step - accuracy: 0.9122 - loss: 0.3332 - val_accuracy: 0.9973 - val_loss: 0.0086
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 197s 420ms/step - accuracy: 0.9749 - loss: 0.0876 - val_accuracy: 0.9973 - val_loss: 0.0081
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 198s 421ms/step - accuracy: 0.9803 - loss: 0.0624 - val_accuracy: 0.9990 - val_loss: 0.0056
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.9850 - loss: 0.0474 - val_accuracy: 0.9983 - val_loss: 0.0056
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 190s 404ms/step - accuracy: 0.9877 - loss: 0.0412 - val_accuracy: 0.9983 - val_loss: 0.0049


Evaluation

In [44]:
loss, acc = model_mobilenet.evaluate(test_data)
print("MobileNetV2 Accuracy:", acc)

import numpy as np
from sklearn.metrics import classification_report

y_pred = np.argmax(model_mobilenet.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.9992 - loss: 0.0069
MobileNetV2 Accuracy: 0.9993333220481873
94/94 ━━━━━━━━━━━━━━━━━━━━ 14s 102ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       0.99      0.99      0.99       200
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       200
           4       1.00      1.00      1.00       200
           5       1.00      1.00      1.00       200
           6       1.00      1.00      1.00       200
           7       1.00      1.00      1.00       200
           8       1.00      1.00      1.00       200
           9       1.00      1.00      1.00       200
          10       1.00      1.00      1.00       200
          11       1.00      1.00      1.00       200
          12       1.00      0.99      1.00       200
          13       1.00      1.00      1.00       200
          14       1.00      1

Overall Performance Comparison

In [45]:
print("Baseline:", baseline_acc, baseline_time)
print("Deep:", deep_acc, deep_time)
print("MobileNetV2:", acc, mn_time)

Baseline: 0.9826666712760925 2671.8814692497253
Deep: 0.7806666493415833 2914.298819065094
MobileNetV2: 0.9993333220481873 1932.579525232315
